> **Prefer Google Colab?** [Open this notebook in Colab](https://colab.research.google.com/github/SpatialTurn/Foundations/blob/main/notebooks/04d_Raster_Data_with_rasterio.ipynb).
> In Colab, click **File > Save a copy in Drive** first so your changes are kept.

# Raster Data with rasterio

**Series:** Geospatial Python Basics (notebook 4 of 4)
**Time:** about 30 minutes

**In this notebook you will:**
- Open a GeoTIFF from the web and read its metadata (CRS, cell size, extent, no-data value)
- Read the cells into a NumPy array, summarize them and plot them
- Look up the value at a point
- Work with a categorical raster (land cover) and count its classes
- Clip a raster to a polygon

**You should already know:** GeoDataFrames and coordinate systems (notebooks 4a and 4b).

> **Before you start:** run each cell in order by clicking it and pressing **Shift + Enter**.

## Rasters in one paragraph

Vector data stores features (points, lines, polygons). **Raster** data covers an area with a grid of equal
cells, and every cell holds one number: an elevation, a temperature, a land cover class, a satellite
reflectance. Four pieces of information turn that grid of numbers into a map:

| Property | Meaning | Example |
|---|---|---|
| **CRS** | The coordinate reference system | EPSG:26916 (UTM zone 16N, meters) |
| **Cell size** (resolution) | The width and height of one cell on the ground | 30 m by 30 m |
| **Extent** (bounds) | Where the grid sits | left, bottom, right, top in meters |
| **No-data value** | The number that means "no measurement here" | -9999 |

**rasterio** is the standard Python library for reading and writing rasters. It reads GeoTIFFs straight from a
web address, just like `gpd.read_file()` reads vector data.

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt

dem_url = "https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_dem.tif"
dem = rasterio.open(dem_url)

print("CRS:       ", dem.crs)
print("Cell size: ", dem.res, "meters")
print("Size:      ", dem.width, "columns by", dem.height, "rows")
print("Bounds:    ", dem.bounds)
print("Bands:     ", dem.count, "| data type:", dem.dtypes[0], "| no-data:", dem.nodata)

This is a **digital elevation model (DEM)** of Tippecanoe County from the USGS 3D Elevation Program: one band,
each cell holding the ground elevation in meters. The cells outside the county are set to the no-data value.

## Read the cells

`read(1)` returns band 1 as a NumPy array: a grid of numbers with rows and columns. With `masked=True`, cells
equal to the no-data value are hidden, so they do not distort statistics.

In [ ]:
elev = dem.read(1, masked=True)
print(elev.shape)                  # (rows, columns)
print("Lowest: ", elev.min(), "m")
print("Highest:", elev.max(), "m")
print("Mean:   ", round(float(elev.mean()), 1), "m")

Try `dem.read(1)` without `masked=True` and print its minimum: you get -9999, the no-data value. Forgetting to
mask no-data is one of the commonest raster mistakes.

## Plot it

`rasterio.plot.show()` draws the array in its real coordinates, so it can share axes with vector layers.

In [ ]:
from rasterio.plot import show
import geopandas as gpd

city = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/CaseStudies/main/data/study_area.geojson").to_crs(dem.crs)

fig, ax = plt.subplots(figsize=(7, 7))
img = show(elev, transform=dem.transform, ax=ax, cmap="terrain")
fig.colorbar(img.get_images()[0], ax=ax, shrink=0.6, label="Elevation (m)")
city.boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_title("Tippecanoe County elevation, with Lafayette and West Lafayette")
plt.show()

The low band running from northeast to southwest is the **Wabash River** valley. Notice that we converted the
city layer to the raster's CRS with `to_crs(dem.crs)`: rasters are usually converted *less* often than
vectors, because reprojecting a raster resamples every cell.

A histogram shows how elevations are distributed. `.compressed()` drops the masked cells.

In [ ]:
plt.hist(elev.compressed(), bins=50, color="tab:brown")
plt.xlabel("Elevation (m)")
plt.ylabel("Number of 30 m cells")
plt.show()

## The value at a point

`sample()` returns the cell values at a list of coordinates. The coordinates must be in the raster's CRS, so we
build a point in latitude and longitude and convert it.

In [ ]:
places = gpd.GeoDataFrame(
    {"name": ["Purdue Memorial Union", "Wabash River at Tapawingo Park"]},
    geometry=gpd.points_from_xy([-86.9107, -86.8988], [40.4247, 40.4237]),
    crs="EPSG:4326",
).to_crs(dem.crs)

coords = [(p.x, p.y) for p in places.geometry]
places["elev_m"] = [v[0] for v in dem.sample(coords)]
places[["name", "elev_m"]]

## A categorical raster: land cover

Not every raster holds measurements. In a **categorical** raster each number is a class code. The National
Land Cover Database (NLCD) uses codes such as 11 (open water), 21 to 24 (developed, from open space to high
intensity), 41 to 43 (forest) and 82 (cultivated crops). Averaging these codes would be meaningless; you count
them instead.

In [ ]:
lc = rasterio.open("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_landcover_2021.tif")
print(lc.crs, lc.res, "| no-data:", lc.nodata)
cover = lc.read(1, masked=True)

codes, counts = np.unique(cover.compressed(), return_counts=True)
names = {11: "Open water", 21: "Developed, open space", 22: "Developed, low intensity",
         23: "Developed, medium intensity", 24: "Developed, high intensity", 31: "Barren",
         41: "Deciduous forest", 42: "Evergreen forest", 43: "Mixed forest", 52: "Shrub",
         71: "Grassland", 81: "Pasture/hay", 82: "Cultivated crops", 90: "Woody wetlands",
         95: "Herbaceous wetlands"}

import pandas as pd
table = pd.DataFrame({"code": codes, "cells": counts})
table["class"] = table["code"].map(names)
table["km2"] = table["cells"] * 30 * 30 / 1e6        # each cell is 30 m by 30 m
table["percent"] = (100 * table["cells"] / table["cells"].sum()).round(1)
table.sort_values("cells", ascending=False)

Cell counts become areas because every cell has the same size: 30 m × 30 m = 900 m². This only works in a CRS
measured in meters, which is why the file was prepared in EPSG:26916.

The file also carries the official NLCD colors, so it can be drawn with them:

In [ ]:
from matplotlib.colors import ListedColormap

cmap_dict = lc.colormap(1)                                  # {code: (r, g, b, a)}
colors = np.array([cmap_dict.get(i, (0, 0, 0, 0)) for i in range(256)]) / 255
fig, ax = plt.subplots(figsize=(7, 7))
show(lc.read(1), transform=lc.transform, ax=ax, cmap=ListedColormap(colors), vmin=0, vmax=255,
     interpolation="nearest")
city.boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_title("Land cover 2021 (NLCD)")
plt.show()

## Clip a raster to a polygon

`rasterio.mask.mask()` keeps the cells inside one or more shapes and sets the rest to no-data. With
`crop=True` it also trims the grid to the shapes' extent. Here we keep only the two cities.

In [ ]:
from rasterio.mask import mask

city_cover, city_transform = mask(lc, city.geometry, crop=True, filled=False)
city_cover = city_cover[0]                     # first (only) band

codes, counts = np.unique(city_cover.compressed(), return_counts=True)
city_table = pd.DataFrame({"code": codes, "cells": counts})
city_table["class"] = city_table["code"].map(names)
city_table["percent"] = (100 * city_table["cells"] / city_table["cells"].sum()).round(1)
city_table.sort_values("cells", ascending=False).head(6)

In the county, cultivated crops dominate; inside the cities, the developed classes do. Summarizing a raster
*inside* polygons like this is called **zonal statistics**, and the Raster Processing episode in the Data Processing
module does it for every census tract at once.

## Try it yourself

1. What share of the **cities' area** is developed (codes 21, 22, 23 and 24 together)?
2. Using the elevation raster, what is the **highest elevation inside the city limits**?

*Hint:* For question 2, use `mask()` on `dem` instead of `lc`.

<details>
<summary><b>Show a solution</b></summary>

```python
developed = city_table[city_table["code"].isin([21, 22, 23, 24])]["percent"].sum()
print(round(developed, 1), "% developed")

city_elev, _ = mask(dem, city.geometry, crop=True, filled=False)
print(city_elev[0].max(), "m")
```

</details>

In [ ]:
# Write your answer here

## Recap

- `rasterio.open(url)` reads a GeoTIFF from the web; `.crs`, `.res`, `.bounds` and `.nodata` describe it
- `.read(1, masked=True)` gives a NumPy array with no-data cells hidden
- `show(array, transform=...)` plots a raster in real coordinates, so vector layers can go on top
- Count the classes of a categorical raster; never average its codes
- `rasterio.mask.mask()` clips a raster to polygons (both in the same CRS)

**Next:** the Data Processing course's Geoprocessing in Python episode (vector operations), or the QGIS episodes
of this module.